# backward-func-lookup — ex1: implement BackwardFuncLookup with (fn, argnum) keys

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `backward-func-lookup`. Running the final beacon cell reports progress against the `Backprop: BackwardFuncLookup` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: BackwardFuncLookup` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`backward-func-lookup`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "backward-func-lookup"
DD_SUBTOPIC = "Backprop: BackwardFuncLookup"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## BackwardFuncLookup — quick refresher

Central registry that maps `(forward_fn, arg_position) -> back_fn`. Tiny by design — it's just a `dict` with two methods:

```python
class BackwardFuncLookup:
    def __init__(self):
        self.back_funcs = {}   # (forward_fn, arg_position) -> back_fn

    def add_back_func(self, forward_fn, arg_position, back_fn):
        self.back_funcs[(forward_fn, arg_position)] = back_fn

    def get_back_func(self, forward_fn, arg_position):
        return self.back_funcs[(forward_fn, arg_position)]
```

Why a 2-key `(fn, argnum)` instead of nested dicts? Same lookup cost, but flatter — registration and dispatch are both `O(1)` one-liners.

Symmetric ops still register TWICE (e.g. `add_back0` and `add_back1` with the same body). The dispatcher in `backprop` does not know which ops are symmetric — it always asks the lookup for `(func, argnum)` given the parent's argnum from `recipe.parents`.

### Exercise 1 — implement BackwardFuncLookup with (fn, argnum) keys

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the (forward_fn, arg_position) → back_fn dispatch pattern by implementing a BackwardFuncLookup class with add and get methods.
> Keywords: backward-func-lookup, dispatcher, register, dict
> ```

**KCs targeted:** `backward-func-lookup`, `register-back-fn-after-wrap`

Implement `BackwardFuncLookup` — the central back-fn dispatcher used by the reverse pass. It's just a `dict` with two methods:

```python
class BackwardFuncLookup:
    def __init__(self):
        self.back_funcs = {}    # (forward_fn, arg_position) -> back_fn

    def add_back_func(self, forward_fn, arg_position, back_fn):
        ...

    def get_back_func(self, forward_fn, arg_position):
        ...
```

Requirements:

1. **`add_back_func(fwd, argnum, back_fn)`** — store `back_fn` keyed by the 2-tuple `(fwd, argnum)`. Overwriting an existing key is fine (makes re-registration cheap).

2. **`get_back_func(fwd, argnum)`** — return the stored back fn. **On a missing key, raise a clear `KeyError`** with a message containing both the function name and the argnum so the user can diagnose missing registrations.

Why the 2-key tuple, not nested dicts? Same lookup cost (`O(1)`), but flat — registration and dispatch are both one-liners. Nested would require `defaultdict(dict)` and an extra `.get` step.

Why both `add` and `get` named methods, not `__setitem__` / `__getitem__`? Named methods make the call sites in the reverse pass self-documenting:

```python
back_fn = BACK_FUNCS.get_back_func(node.recipe.func, argnum)
```

vs the cryptic `BACK_FUNCS[(node.recipe.func, argnum)]`. Both work; the named methods are the convention.

In [ ]:
class BackwardFuncLookup:
    def __init__(self):
        # Flat dict keyed by (forward_fn, arg_position).
        self.back_funcs = {}

    def add_back_func(self, forward_fn, arg_position, back_fn):
        self.back_funcs[(forward_fn, arg_position)] = back_fn

    def get_back_func(self, forward_fn, arg_position):
        key = (forward_fn, arg_position)
        if key not in self.back_funcs:
            raise KeyError(
                f'No back_fn registered for ({forward_fn!r}, argnum={arg_position}). '
                f'Did you forget BACK_FUNCS.add_back_func({forward_fn.__name__}, '
                f'{arg_position}, ...)?'
            )
        return self.back_funcs[key]


<details><summary>Solution</summary>

```python
class BackwardFuncLookup:
    def __init__(self):
        # Flat dict keyed by (forward_fn, arg_position).
        self.back_funcs = {}

    def add_back_func(self, forward_fn, arg_position, back_fn):
        self.back_funcs[(forward_fn, arg_position)] = back_fn

    def get_back_func(self, forward_fn, arg_position):
        key = (forward_fn, arg_position)
        if key not in self.back_funcs:
            raise KeyError(
                f'No back_fn registered for ({forward_fn!r}, argnum={arg_position}). '
                f'Did you forget BACK_FUNCS.add_back_func({forward_fn.__name__}, '
                f'{arg_position}, ...)?'
            )
        return self.back_funcs[key]
```

**Why `KeyError` with a diagnostic message.** The default `self.back_funcs[key]` would also raise `KeyError`, but the message would be `KeyError: (<function torch.log>, 0)` — opaque. Spelling out 'No back_fn registered for ... Did you forget add_back_func(...)?' turns a 5-minute hunt into a one-second fix. Worth the three-line investment.

**Why the flat 2-tuple key.** Conceptually we want a two-dimensional lookup `(fn, argnum) -> back_fn`. Three natural implementations:
- Nested dict `{fn: {argnum: back_fn}}` — needs `defaultdict` + two `.get` calls.
- Flat 2-tuple key (this one) — one dict, one access.
- Class attribute on the back fn itself (e.g. `back_fn._for = (fn, 0)`) — clever, but the registry is global anyway, so the class isn't carrying its weight.

Flat 2-tuple wins on simplicity.

**Two instances → two registries.** A single global `BACK_FUNCS` is the usual setup, but having `__init__` create a per-instance dict means you can spin up a separate registry for unit tests without polluting the global. The test exercises this.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()